
# MathVision Hybrid-Lite Matchmaking Notebook

This notebook builds a tutor-recommendation prototype driven by provided CSV files.
It combines:
- a transparent rule-based score,
- a simple Logistic Regression success predictor,
- and a final hybrid score for ranking.



## 1) Business context

The notebook now uses real project-style input files instead of synthetic generation.


## 2) Imports and setup


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATHS = {
    "students": "mathvision_students_named.csv",
    "tutors": "mathvision_tutors_named.csv",
    "historical_matches": "mathvision_historical_matches_named.csv",
}

TOPICS = ["Algebra", "Fractions", "Geometry", "Calculus", "Statistics"]
CURRICULA = ["Local", "IB", "IGCSE"]


## 3) Preprocessing Module (Spec A)

This section converts raw tutor feedback notes into `pairings_labeled.csv` using text cleaning, TF-IDF, cosine similarity, and pairing-level aggregation.


In [ ]:
# Preprocessing input/output parameters
PREPROCESSING_PATHS = {
    "students": "students.csv",
    "tutors": "tutors.csv",
    "pairings_raw": "pairings_raw.csv",
    "pairings_labeled": "pairings_labeled.csv",
}

POSITIVE_PHRASES = [
    "improving understanding",
    "good progress",
    "grasped the concept",
    "more confident",
    "able to solve independently",
    "ready for next topic",
]

NEGATIVE_PHRASES = [
    "still struggling",
    "weak understanding",
    "confused about topic",
    "many mistakes",
    "needs repeated explanation",
    "unable to solve independently",
]

REQUIRED_RAW_COLUMNS = [
    "pairing_id",
    "student_id",
    "tutor_id",
    "session_date",
    "duration_hours",
    "tutor_feedback_text",
]

PAIRINGS_LABELED_COLUMNS = [
    "pairing_id",
    "student_id",
    "tutor_id",
    "lessons_count",
    "total_hours",
    "avg_feedback_score",
    "positive_note_count",
    "negative_note_count",
    "good_pairing_label",
]


In [ ]:
def clean_feedback_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def validate_pairings_raw(df_raw):
    missing_cols = [c for c in REQUIRED_RAW_COLUMNS if c not in df_raw.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    for col in ["pairing_id", "student_id", "tutor_id", "tutor_feedback_text"]:
        if df_raw[col].isna().any():
            bad_idx = int(df_raw[df_raw[col].isna()].index[0])
            raise ValueError(f"Column {col} has null at row index {bad_idx}")

    duration_numeric = pd.to_numeric(df_raw["duration_hours"], errors="coerce")
    if duration_numeric.isna().any():
        bad_idx = int(duration_numeric[duration_numeric.isna()].index[0])
        bad_val = df_raw.loc[bad_idx, "duration_hours"]
        raise ValueError(f"duration_hours is non-numeric at row index {bad_idx}: {bad_val}")
    if (duration_numeric < 0).any():
        bad_idx = int(duration_numeric[duration_numeric < 0].index[0])
        bad_val = float(duration_numeric.loc[bad_idx])
        raise ValueError(f"duration_hours is negative at row index {bad_idx}: {bad_val}")


def build_reference_vectors(positive_phrases, negative_phrases, vectorizer=None):
    if vectorizer is None:
        vectorizer = TfidfVectorizer()

    cleaned_positive = [clean_feedback_text(p) for p in positive_phrases]
    cleaned_negative = [clean_feedback_text(p) for p in negative_phrases]
    vectorizer.fit(cleaned_positive + cleaned_negative)

    positive_reference_vector = vectorizer.transform([" ".join(cleaned_positive)])
    negative_reference_vector = vectorizer.transform([" ".join(cleaned_negative)])
    return vectorizer, positive_reference_vector, negative_reference_vector


def score_session_feedback(tutor_feedback_text, vectorizer, positive_reference_vector, negative_reference_vector):
    cleaned_note = clean_feedback_text(tutor_feedback_text)
    note_vector = vectorizer.transform([cleaned_note])

    similarity_positive = float(cosine_similarity(note_vector, positive_reference_vector)[0][0])
    similarity_negative = float(cosine_similarity(note_vector, negative_reference_vector)[0][0])
    feedback_score = similarity_positive - similarity_negative

    return similarity_positive, similarity_negative, feedback_score


def label_pairing(avg_feedback_score):
    return 1 if avg_feedback_score > 0 else 0


def aggregate_pairings(scored_sessions_df):
    grouped = (
        scored_sessions_df
        .groupby(["pairing_id", "student_id", "tutor_id"], as_index=False)
        .agg(
            lessons_count=("pairing_id", "size"),
            total_hours=("duration_hours", "sum"),
            avg_feedback_score=("feedback_score", "mean"),
            positive_note_count=("feedback_score", lambda s: int((s > 0).sum())),
            negative_note_count=("feedback_score", lambda s: int((s < 0).sum())),
        )
    )
    grouped["good_pairing_label"] = grouped["avg_feedback_score"].apply(label_pairing)
    return grouped[PAIRINGS_LABELED_COLUMNS]


def preprocess_pairings_raw(df_raw, positive_phrases=None, negative_phrases=None):
    if positive_phrases is None:
        positive_phrases = POSITIVE_PHRASES
    if negative_phrases is None:
        negative_phrases = NEGATIVE_PHRASES

    validate_pairings_raw(df_raw)
    df = df_raw.copy()
    df["duration_hours"] = pd.to_numeric(df["duration_hours"], errors="raise")
    df["cleaned_feedback_text"] = df["tutor_feedback_text"].apply(clean_feedback_text)

    vectorizer, positive_ref_vec, negative_ref_vec = build_reference_vectors(
        positive_phrases=positive_phrases,
        negative_phrases=negative_phrases,
    )

    scored_rows = df["cleaned_feedback_text"].apply(
        lambda txt: score_session_feedback(txt, vectorizer, positive_ref_vec, negative_ref_vec)
    )
    scores_df = pd.DataFrame(
        scored_rows.tolist(),
        columns=["similarity_positive", "similarity_negative", "feedback_score"],
        index=df.index,
    )
    df = pd.concat([df, scores_df], axis=1)

    labeled_df = aggregate_pairings(df)
    return labeled_df, df


def run_preprocessing_from_csv(
    pairings_raw_path,
    output_path,
    positive_phrases=None,
    negative_phrases=None,
):
    df_raw = pd.read_csv(pairings_raw_path)
    labeled_df, scored_sessions_df = preprocess_pairings_raw(
        df_raw=df_raw,
        positive_phrases=positive_phrases,
        negative_phrases=negative_phrases,
    )
    labeled_df.to_csv(output_path, index=False)
    return labeled_df, scored_sessions_df


### Preprocessing Checks

These checks validate cleaning, scoring directionality, aggregation behavior, and strict-validation failures.


In [ ]:
# Unit-like checks for cleaning and score direction
assert clean_feedback_text("Student showed Improvement in Fractions!") == "student showed improvement in fractions"

vectorizer, pos_ref, neg_ref = build_reference_vectors(POSITIVE_PHRASES, NEGATIVE_PHRASES)
_, _, pos_score = score_session_feedback(
    "Student made good progress and is more confident.",
    vectorizer,
    pos_ref,
    neg_ref,
)
_, _, neg_score = score_session_feedback(
    "Student is still struggling and unable to solve independently.",
    vectorizer,
    pos_ref,
    neg_ref,
)
assert pos_score > 0
assert neg_score < 0
print("Cleaning and score direction checks passed.")


In [ ]:
# Aggregation and label checks with toy sessions
toy_df = pd.DataFrame(
    [
        {
            "pairing_id": "P001",
            "student_id": "S001",
            "tutor_id": "T001",
            "session_date": "2026-03-01",
            "duration_hours": 1.5,
            "tutor_feedback_text": "Good progress and improving understanding.",
        },
        {
            "pairing_id": "P001",
            "student_id": "S001",
            "tutor_id": "T001",
            "session_date": "2026-03-08",
            "duration_hours": 1.0,
            "tutor_feedback_text": "Still struggling with many mistakes.",
        },
        {
            "pairing_id": "P002",
            "student_id": "S002",
            "tutor_id": "T002",
            "session_date": "2026-03-02",
            "duration_hours": 2.0,
            "tutor_feedback_text": "Grasped the concept and ready for next topic.",
        },
    ]
)

toy_labeled, toy_scored = preprocess_pairings_raw(toy_df)

p001 = toy_labeled[toy_labeled["pairing_id"] == "P001"].iloc[0]
assert int(p001["lessons_count"]) == 2
assert abs(float(p001["total_hours"]) - 2.5) < 1e-9
assert int(p001["positive_note_count"] + p001["negative_note_count"]) <= int(p001["lessons_count"])
assert int(p001["good_pairing_label"]) == (1 if float(p001["avg_feedback_score"]) > 0 else 0)

assert list(toy_labeled.columns) == PAIRINGS_LABELED_COLUMNS
print("Aggregation and labeling checks passed.")


In [ ]:
# Strict validation checks (expected failures)
missing_col_df = pd.DataFrame(
    [{
        "pairing_id": "P003",
        "student_id": "S003",
        "tutor_id": "T003",
        "session_date": "2026-03-03",
        "duration_hours": 1.0,
    }]
)
try:
    validate_pairings_raw(missing_col_df)
    raise AssertionError("Expected missing-column validation failure")
except ValueError as exc:
    assert "Missing required columns" in str(exc)

null_feedback_df = pd.DataFrame(
    [{
        "pairing_id": "P004",
        "student_id": "S004",
        "tutor_id": "T004",
        "session_date": "2026-03-04",
        "duration_hours": 1.0,
        "tutor_feedback_text": None,
    }]
)
try:
    validate_pairings_raw(null_feedback_df)
    raise AssertionError("Expected null-feedback validation failure")
except ValueError as exc:
    assert "tutor_feedback_text" in str(exc)

invalid_duration_df = pd.DataFrame(
    [{
        "pairing_id": "P005",
        "student_id": "S005",
        "tutor_id": "T005",
        "session_date": "2026-03-05",
        "duration_hours": "bad",
        "tutor_feedback_text": "Good progress.",
    }]
)
try:
    validate_pairings_raw(invalid_duration_df)
    raise AssertionError("Expected duration validation failure")
except ValueError as exc:
    assert "duration_hours" in str(exc)

print("Strict validation checks passed.")


In [ ]:
# Optional run block: executes once pairings_raw.csv is provided
raw_path = Path(PREPROCESSING_PATHS["pairings_raw"])
if raw_path.exists():
    labeled_df, scored_df = run_preprocessing_from_csv(
        pairings_raw_path=PREPROCESSING_PATHS["pairings_raw"],
        output_path=PREPROCESSING_PATHS["pairings_labeled"],
    )
    print(f"Saved {len(labeled_df)} rows to {PREPROCESSING_PATHS['pairings_labeled']}")
    print(labeled_df.head())
else:
    print(
        f"Skipped full preprocessing run: {PREPROCESSING_PATHS['pairings_raw']} not found. "
        "Provide the raw file, then rerun this cell."
    )


## 4) Load data from CSV files


In [ ]:
def parse_slots(slot_value):
    if pd.isna(slot_value):
        return []
    if isinstance(slot_value, list):
        return slot_value
    return [slot.strip() for slot in str(slot_value).split(";") if slot.strip()]


def load_data_and_prepare(path_students, path_tutors, path_matches):
    students_df = pd.read_csv(path_students)
    tutors_df = pd.read_csv(path_tutors)
    historical_matches_df = pd.read_csv(path_matches)

    tutors_df["available_slots"] = tutors_df["available_slots"].apply(parse_slots)

    # Required numeric fields as numeric
    numeric_cols_students = ["grade_level"]
    students_df[numeric_cols_students] = students_df[numeric_cols_students].apply(pd.to_numeric, errors="coerce")

    numeric_cols_tutors = [
        "years_experience",
        "rating",
        "past_success_rate",
        "preferred_min_grade",
        "preferred_max_grade",
    ]
    tutors_df[numeric_cols_tutors] = tutors_df[numeric_cols_tutors].apply(
        pd.to_numeric, errors="coerce"
    )

    # Keep historical match columns numeric and ensure binary flags are numeric ints
    historical_matches_df[[
        "topic_match",
        "curriculum_match",
        "availability_match",
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
        "success_label",
    ]] = historical_matches_df[[
        "topic_match",
        "curriculum_match",
        "availability_match",
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
        "success_label",
    ]].apply(pd.to_numeric, errors="coerce")

    # Report any missing values to keep notebook deterministic and debuggable
    print("students_df shape:", students_df.shape)
    print("tutors_df shape:", tutors_df.shape)
    print("historical_matches_df shape:", historical_matches_df.shape)
    print("Missing values")
    print("students_df:", students_df.isna().sum().sum())
    print("tutors_df:", tutors_df.isna().sum().sum())
    print("historical_matches_df:", historical_matches_df.isna().sum().sum())

    return students_df, tutors_df, historical_matches_df


students_df, tutors_df, historical_matches_df = load_data_and_prepare(
    DATA_PATHS["students"],
    DATA_PATHS["tutors"],
    DATA_PATHS["historical_matches"],
)

print(students_df.head())
print(tutors_df.head())
print(historical_matches_df.head())



## 4) Exploratory analysis


In [ ]:

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
axs = axs.ravel()

students_df["curriculum"].value_counts().sort_index().plot(kind="bar", ax=axs[0], title="Students by curriculum")
axs[0].set_ylabel("count")

tutors_df["specialty_topic"].value_counts().sort_index().plot(kind="bar", ax=axs[1], title="Tutors by specialty topic")
axs[1].set_ylabel("count")

axs[2].hist(tutors_df["rating"], bins=10)
axs[2].set_title("Tutor rating histogram")
axs[2].set_xlabel("rating")
axs[2].set_ylabel("count")

historical_matches_df["success_label"].value_counts().sort_index().plot(
    kind="bar", ax=axs[3], title="Success label distribution"
)
axs[3].set_xlabel("success_label")
axs[3].set_ylabel("count")

plt.tight_layout()



## 5) Rule-based scoring helper

The matching logic follows the project spec.


In [ ]:

def compute_rule_features(student_row, tutor_row):
    topic_match = int(tutor_row["specialty_topic"] == student_row["weak_topic"])
    curriculum_match = int(tutor_row["primary_curriculum"] == student_row["curriculum"])
    availability_match = int(student_row["requested_slot"] in tutor_row["available_slots"])

    grade_center = (float(tutor_row["preferred_min_grade"]) + float(tutor_row["preferred_max_grade"])) / 2
    grade_gap = abs(float(student_row["grade_level"]) - grade_center)

    tutor_rating = float(tutor_row["rating"])
    tutor_experience = int(tutor_row["years_experience"])
    past_success_rate = float(tutor_row["past_success_rate"])

    match_score_rule = (
        0.35 * topic_match
        + 0.25 * curriculum_match
        + 0.20 * (tutor_rating / 5)
        + 0.10 * availability_match
        + 0.10 * past_success_rate
    )

    return {
        "topic_match": topic_match,
        "curriculum_match": curriculum_match,
        "availability_match": availability_match,
        "grade_gap": grade_gap,
        "tutor_rating": tutor_rating,
        "tutor_experience": tutor_experience,
        "past_success_rate": past_success_rate,
        "match_score_rule": float(match_score_rule),
    }


sample_student = students_df.iloc[0]
sample_tutor = tutors_df.iloc[0]
sample_features = compute_rule_features(sample_student, sample_tutor)

print("Sample student:", sample_student["student_id"], sample_student["curriculum"], sample_student["weak_topic"], sample_student["requested_slot"])
print("Sample tutor:", sample_tutor["tutor_id"], sample_tutor["primary_curriculum"], sample_tutor["specialty_topic"])
print("Computed features + rule score:")
for k, v in sample_features.items():
    print(f"{k}: {v}")



## 6) Train logistic regression success model


In [ ]:

def train_success_model(train_df):
    feature_cols = [
        "topic_match",
        "curriculum_match",
        "availability_match",
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
    ]

    numeric_cols = [
        "grade_gap",
        "tutor_rating",
        "tutor_experience",
        "past_success_rate",
        "match_score_rule",
    ]

    X = train_df[feature_cols].copy()
    y = train_df["success_label"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_scaled[numeric_cols])
    X_test_scaled[numeric_cols] = scaler.transform(X_test_scaled[numeric_cols])

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"F1-score:  {f1:.3f}")

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix")
    plt.tight_layout()

    return model, scaler, feature_cols


log_reg_model, scaler, FEATURE_COLUMNS = train_success_model(historical_matches_df)



## 7) Ranking engine

Build ranked recommendations for a student by combining rule score + ML probability.


In [ ]:

def _build_explanation(features_row):
    reasons = []
    if features_row["topic_match"] == 1:
        reasons.append("Strong topic match")
    if features_row["curriculum_match"] == 1:
        reasons.append("Curriculum match")
    if features_row["availability_match"] == 1:
        reasons.append("Available in requested slot")
    if features_row["tutor_rating"] >= 4.6:
        reasons.append("High tutor rating")
    if features_row["past_success_rate"] >= 0.8:
        reasons.append("Good past success rate")
    if not reasons:
        return "No direct rule match flags; model probability still contributes"
    return " | ".join(reasons)


def rank_tutors_for_student(student_row, tutors_df, model, scaler, feature_cols):
    feature_rows = []
    for _, tutor_row in tutors_df.iterrows():
        features = compute_rule_features(student_row, tutor_row)
        feature_rows.append(
            {
                "tutor_id": tutor_row["tutor_id"],
                "specialty_topic": tutor_row["specialty_topic"],
                "primary_curriculum": tutor_row["primary_curriculum"],
                "rating": tutor_row["rating"],
                "years_experience": tutor_row["years_experience"],
                **features,
            }
        )

    ranking_df = pd.DataFrame(feature_rows)
    numeric_cols = ["grade_gap", "tutor_rating", "tutor_experience", "past_success_rate", "match_score_rule"]
    X = ranking_df[feature_cols].copy()
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler.transform(X_scaled[numeric_cols])

    ranking_df["predicted_success_probability"] = model.predict_proba(X_scaled)[:, 1]
    ranking_df["final_score"] = (
        0.6 * ranking_df["match_score_rule"] +
        0.4 * ranking_df["predicted_success_probability"]
    )
    ranking_df["rule_score"] = ranking_df["match_score_rule"]
    ranking_df["explanation"] = ranking_df.apply(_build_explanation, axis=1)

    ranking_df = ranking_df.sort_values("final_score", ascending=False).reset_index(drop=True)
    return ranking_df[[
        "tutor_id",
        "specialty_topic",
        "primary_curriculum",
        "rating",
        "years_experience",
        "rule_score",
        "predicted_success_probability",
        "final_score",
        "explanation",
    ]]


demo_student = students_df.sample(1, random_state=42).iloc[0]
demo_ranked = rank_tutors_for_student(
    demo_student,
    tutors_df,
    log_reg_model,
    scaler,
    FEATURE_COLUMNS,
)
print("Demo student:", demo_student["student_id"], demo_student["student_name"], demo_student["curriculum"], demo_student["weak_topic"])
display(demo_ranked.head(5))

top5 = demo_ranked.head(5)
plt.figure(figsize=(8, 4))
plt.barh(top5["tutor_id"], top5["final_score"], color="#2c7fb8")
plt.gca().invert_yaxis()
plt.title("Top 5 tutors for one student")
plt.xlabel("final_score")
plt.tight_layout()



## 8) Demo scenarios

Run the ranking for the three requested student profiles.


In [ ]:

def pick_student(curriculum, weak_topic):
    matches = students_df[(students_df["curriculum"] == curriculum) & (students_df["weak_topic"] == weak_topic)]
    if len(matches) == 0:
        print(f"No student found for {curriculum} + {weak_topic}; using first student as fallback")
        return students_df.iloc[0]
    return matches.iloc[0]

scenarios = [
    ("Local", "Fractions"),
    ("IB", "Calculus"),
    ("IGCSE", "Geometry"),
]

for idx, (curriculum, topic) in enumerate(scenarios, start=1):
    student_row = pick_student(curriculum, topic)
    print(f"Scenario {idx}: {student_row['student_id']} | {curriculum} | {topic}")
    ranked = rank_tutors_for_student(student_row, tutors_df, log_reg_model, scaler, FEATURE_COLUMNS)
    display(ranked.head(5))
    print("-" * 70)



## 9) Business interpretation

This flow is stronger than random tutor assignment because scoring is explicit and consistent,
while predictions learn from historical pair performance.



## 10) Limitations and future improvements

- Inputs now come from provided CSV files, not synthetic generation.
- Historical outcomes depend on source data quality and may be biased.
- `available_slots` parsing assumes semi-colon separated slots.
- Future improvements: add retention/satisfaction features and scheduling constraints.
